# British Airways Heathrow Lounge Demand Analysis

## Project Overview

This project develops a data-driven lookup table to estimate the number of passengers eligible for British Airways lounge tiers at Heathrow Terminal 3.

Using historical flight schedule data, the analysis examines whether haul type, geographical region, or time of day provides the most useful basis for estimating lounge eligibility.

The project will also evaluate monthly stability and test the lookup table on a holdout sample to assess its usefulness for airport planning.

## Objectives

1. Inspect and validate the flight schedule dataset.
2. Calculate lounge eligibility rates for Tiers 1, 2, and 3.
3. Compare alternative flight groupings.
4. Evaluate the stability of eligibility rates over time.
5. Test the lookup table against an unseen sample.
6. Translate the findings into practical airport-planning recommendations.

## Important Assumption

The dataset contains seat counts rather than actual passenger counts. Seat-based estimates therefore assume full occupancy unless a different load factor is explicitly introduced.

Tier 1 estimates represent hypothetical Concorde Room demand; Heathrow Terminal 3 does not have a Concorde Room.

# Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries imported successfully.")


# **Load Dataset**

In [ ]:
# Portable loading: works in Colab (upload prompt) and locally (file in the same folder)
import os
DATA_PATH = "British Airways Summer Schedule Dataset - Forage Data Science Task 1.xlsx"

if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files   # only exists in Colab
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Put the dataset next to this notebook as: {DATA_PATH}")

df = pd.read_excel(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Number of flights: {len(df):,}")
print(f"Number of columns: {len(df.columns)}")


# **Inspect the dataset**

In [ ]:

# View the first five rows
display(df.head())




In [ ]:
# Inspect column names and data types
print("\nDataset information:")
df.info()



In [ ]:
# Check missing values
print("\nMissing values per column:")
display(df.isnull().sum().to_frame("Missing Values"))

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# **Check the data**

In [ ]:

# 1. Check date range
print("DATE RANGE")
print("Start:", df["FLIGHT_DATE"].min())
print("End:  ", df["FLIGHT_DATE"].max())

# 2. Check number of unique destinations and categories
print("\nDATASET COVERAGE")
print("Unique destinations:", df["ARRIVAL_STATION_CD"].nunique())
print("Countries:", df["ARRIVAL_COUNTRY"].nunique())
print("Regions:", df["ARRIVAL_REGION"].nunique())
print("Haul types:", df["HAUL"].unique())
print("Time categories:", df["TIME_OF_DAY"].unique())
print("Aircraft types:", df["AIRCRAFT_TYPE"].nunique())




In [ ]:
# 3. Check seat counts
seat_columns = [
    "FIRST_CLASS_SEATS",
    "BUSINESS_CLASS_SEATS",
    "ECONOMY_SEATS"
]

df["TOTAL_SEATS"] = df[seat_columns].sum(axis=1)

print("\nSEAT CHECKS")
print("Flights with zero seats:", (df["TOTAL_SEATS"] == 0).sum())
print("Minimum seats per flight:", df["TOTAL_SEATS"].min())
print("Maximum seats per flight:", df["TOTAL_SEATS"].max())

# 4. Check eligibility counts
tier_columns = [
    "TIER1_ELIGIBLE_PAX",
    "TIER2_ELIGIBLE_PAX",
    "TIER3_ELIGIBLE_PAX"
]

print("\nELIGIBILITY CHECKS")
print("Negative eligibility counts:",
      (df[tier_columns] < 0).any(axis=1).sum())

print("Tier 1 eligible passengers:", df["TIER1_ELIGIBLE_PAX"].sum())
print("Tier 2 eligible passengers:", df["TIER2_ELIGIBLE_PAX"].sum())
print("Tier 3 eligible passengers:", df["TIER3_ELIGIBLE_PAX"].sum())

display(df[["TOTAL_SEATS"] + tier_columns].describe().round(2))

# **Calculate the first eligibility rates**

In [ ]:

# Calculate eligible passengers as a share of total seats
haul_summary = (
    df.groupby("HAUL")
    .agg(
        Flights=("FLIGHT_NO", "count"),
        Total_Seats=("TOTAL_SEATS", "sum"),
        Tier1_Pax=("TIER1_ELIGIBLE_PAX", "sum"),
        Tier2_Pax=("TIER2_ELIGIBLE_PAX", "sum"),
        Tier3_Pax=("TIER3_ELIGIBLE_PAX", "sum")
    )
)

# Convert counts into percentages of seats
for tier in ["Tier1", "Tier2", "Tier3"]:
    haul_summary[f"{tier}_Rate_%"] = (
        haul_summary[f"{tier}_Pax"]
        / haul_summary["Total_Seats"] * 100
    )

display(haul_summary.round(2))


**One important finding:**

short-haul flights have
higher eligibility rates across all three tiers, even though long-haul flights have more premium-cabin seats proportionally.

In [ ]:
# ADDED: evidence for the finding above (premium-cabin share of seats by haul)
df["PREMIUM_SEATS"] = df["FIRST_CLASS_SEATS"] + df["BUSINESS_CLASS_SEATS"]
premium = df.groupby("HAUL")[["PREMIUM_SEATS", "TOTAL_SEATS"]].sum()
premium["Premium_share_%"] = premium["PREMIUM_SEATS"] / premium["TOTAL_SEATS"] * 100
display(premium.round(2))

# Eligibility is not only about cabin: Tier 3 exceeds the number of business seats on many flights,
# so loyalty-status customers (for example in economy) must be part of the count.
print("Flights where Tier 3 eligible pax exceed business-class seats:",
      (df["TIER3_ELIGIBLE_PAX"] > df["BUSINESS_CLASS_SEATS"]).sum())

# **Compare haul, region and time of day**

In [ ]:

# Function to calculate weighted eligibility rates for any grouping
def calculate_rates(group_column):
    summary = df.groupby(group_column).agg(
        Flights=("FLIGHT_NO", "count"),
        Total_Seats=("TOTAL_SEATS", "sum"),
        Tier1_Pax=("TIER1_ELIGIBLE_PAX", "sum"),
        Tier2_Pax=("TIER2_ELIGIBLE_PAX", "sum"),
        Tier3_Pax=("TIER3_ELIGIBLE_PAX", "sum")
    )

    for tier in ["Tier1", "Tier2", "Tier3"]:
        summary[f"{tier}_Rate_%"] = (
            summary[f"{tier}_Pax"] / summary["Total_Seats"] * 100
        )

    # CAUTION: this adds the three tiers together. The tier checks later in this notebook
    # show the tiers are not nested and may overlap, so treat this column as descriptive only.
    summary["Sum_of_Tiers_%"] = (
        summary[["Tier1_Pax", "Tier2_Pax", "Tier3_Pax"]].sum(axis=1)
        / summary["Total_Seats"] * 100
    )

    return summary   # round only when displaying


print("RATES BY REGION")
display(calculate_rates("ARRIVAL_REGION"))

print("RATES BY TIME OF DAY")
display(calculate_rates("TIME_OF_DAY"))

print("RATES BY HAUL AND TIME OF DAY")
display(calculate_rates(["HAUL", "TIME_OF_DAY"]))

print("RATES BY HAUL AND REGION")
display(calculate_rates(["HAUL", "ARRIVAL_REGION"]))


# **Check monthly stability**

In [ ]:

df["MONTH"] = df["FLIGHT_DATE"].dt.to_period("M").astype(str)

monthly = df.groupby(["MONTH", "HAUL"]).agg(
    Flights=("FLIGHT_NO", "count"),
    Total_Seats=("TOTAL_SEATS", "sum"),
    Tier1_Pax=("TIER1_ELIGIBLE_PAX", "sum"),
    Tier2_Pax=("TIER2_ELIGIBLE_PAX", "sum"),
    Tier3_Pax=("TIER3_ELIGIBLE_PAX", "sum")
).reset_index()

for tier in ["Tier1", "Tier2", "Tier3"]:
    monthly[f"{tier}_Rate_%"] = (
        monthly[f"{tier}_Pax"] / monthly["Total_Seats"] * 100
    )

display(monthly.round(3))

# Plot total tier eligibility rates by month and haul
for haul in sorted(df["HAUL"].unique()):
    subset = monthly[monthly["HAUL"] == haul]

    plt.figure(figsize=(9, 4))
    for tier in ["Tier1", "Tier2", "Tier3"]:
        plt.plot(
            subset["MONTH"],
            subset[f"{tier}_Rate_%"],
            marker="o",
            label=tier
        )

    plt.title(f"Monthly Eligibility Rates: {haul.title()}-Haul")
    plt.xlabel("Month")
    plt.ylabel("Eligible passengers as % of seats")
    plt.xticks(rotation=45)
    plt.legend()
    plt.tight_layout()
    plt.show()


**Region**: *Within long-haul, Tier 3 eligibility ranges from 10.29% (Middle East) to 10.55% (North America), a spread of only 0.26 percentage points, and Tier 2 sits between about 2.7% and 2.8%. Region does not appear to earn a separate row.*

**Time of day**: *Within short-haul, Tier 3 eligibility ranges from 16.59% (afternoon) to 17.28% (lunchtime), a spread of 0.69 percentage points; within long-haul it ranges from 10.26% (evening) to 10.63% (morning), a spread of 0.37 points. These differences are modest, so time of day is better retained for forecasting when lounge demand arrives, not for changing the eligibility percentages.*

**Combined groupings**: *The same pattern holds when we compare region and time within each haul category. A two-row lookup table is the simpler choice.*

**NOTE:** the `Sum_of_Tiers_%` column in the tables above adds the three tiers together. The tier checks below show the tiers are not nested and may overlap, so treat that column as descriptive only and compare Tier 1, Tier 2 and Tier 3 separately.

In [ ]:
# Check whether tier counts are nested on every flight
tier_order_check = pd.DataFrame({
    "Tier1 <= Tier2": (
        df["TIER1_ELIGIBLE_PAX"] <= df["TIER2_ELIGIBLE_PAX"]
    ),
    "Tier2 <= Tier3": (
        df["TIER2_ELIGIBLE_PAX"] <= df["TIER3_ELIGIBLE_PAX"]
    )
})

print(tier_order_check.sum())
print("\nFlights violating either condition:")
display(df.loc[~tier_order_check.all(axis=1), [
    "FLIGHT_NO",
    "TIER1_ELIGIBLE_PAX",
    "TIER2_ELIGIBLE_PAX",
    "TIER3_ELIGIBLE_PAX"
]].head(10))

# **Inspect the relationships between the columns**

In [ ]:

# Compare the three eligibility columns
print("Tier 1 equals Tier 2:")
print((df["TIER1_ELIGIBLE_PAX"] == df["TIER2_ELIGIBLE_PAX"]).sum())

print("\nTier 2 equals Tier 3:")
print((df["TIER2_ELIGIBLE_PAX"] == df["TIER3_ELIGIBLE_PAX"]).sum())

print("\nTier 1 exceeds Tier 2:")
print((df["TIER1_ELIGIBLE_PAX"] > df["TIER2_ELIGIBLE_PAX"]).sum())

print("\nTier 2 exceeds Tier 3:")
print((df["TIER2_ELIGIBLE_PAX"] > df["TIER3_ELIGIBLE_PAX"]).sum())

print("\nFlights where Tier 1 exceeds Tier 2:")
display(
    df.loc[
        df["TIER1_ELIGIBLE_PAX"] > df["TIER2_ELIGIBLE_PAX"],
        [
            "HAUL",
            "FIRST_CLASS_SEATS",
            "BUSINESS_CLASS_SEATS",
            "ECONOMY_SEATS",
            "TIER1_ELIGIBLE_PAX",
            "TIER2_ELIGIBLE_PAX",
            "TIER3_ELIGIBLE_PAX"
        ]
    ].head(15)
)


# **Extra tier check (is Tier 1 contained in Tier 3?)**

In [ ]:
print("Tier 1 exceeds Tier 3:", (df["TIER1_ELIGIBLE_PAX"] > df["TIER3_ELIGIBLE_PAX"]).sum(), "flights")
print("Tier 1 exceeds Tier 2:", (df["TIER1_ELIGIBLE_PAX"] > df["TIER2_ELIGIBLE_PAX"]).sum(), "flights")
print("Tier 2 exceeds Tier 3:", (df["TIER2_ELIGIBLE_PAX"] > df["TIER3_ELIGIBLE_PAX"]).sum(), "flights")
print("Sum of the three tiers exceeds total seats:",
      (df[tier_columns].sum(axis=1) > df["TOTAL_SEATS"]).sum(), "flights")


Tier 1 eligibility counts exceed Tier 2 on 459 flights and Tier 3 on 126 flights, indicating that the tiers do not follow a strictly nested structure in the observed data. However, these results do not establish whether passengers overlap between tiers. The combined counts never exceed total seats, but this alone does not prove that the tiers are mutually exclusive

# **Is time of day a real effect? (corrected significance tests)**

A plain binomial GLM treats every seat as an independent trial and gives p-values that are far too small here: eligible counts vary about 3 times more between flights than that assumption allows (dispersion 1.7 to 11.5). The tests below allow for that extra variation. A p-value above 0.05 means no reliable difference, not proof of no difference.

In [ ]:
from scipy import stats

def holm(pvals):
    p = np.asarray(pvals, float)
    order = np.argsort(p)
    adj, running, m = np.empty_like(p), 0.0, len(p)
    for rank, idx in enumerate(order):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj

rng = np.random.default_rng(42)
TIER_COLS = {"Tier 1": "TIER1_ELIGIBLE_PAX", "Tier 2": "TIER2_ELIGIBLE_PAX", "Tier 3": "TIER3_ELIGIBLE_PAX"}
rows = []
for haul, g in df.groupby("HAUL"):
    n = g["TOTAL_SEATS"].values.astype(float)
    codes0, _ = pd.factorize(g["TIME_OF_DAY"])
    k = codes0.max() + 1
    for tier, col in TIER_COLS.items():
        y = g[col].values.astype(float)
        p0 = y.sum() / n.sum()
        # 1) ANOVA on flight-level rates (for comparison)
        rate = y / n
        _, p_anova = stats.f_oneway(*[rate[codes0 == j] for j in range(k)])
        # 2) quasi-binomial: binomial likelihood-ratio statistic divided by the dispersion
        ys = np.bincount(codes0, weights=y, minlength=k)
        ns = np.bincount(codes0, weights=n, minlength=k)
        pm = np.clip(ys / ns, 1e-12, 1 - 1e-12)[codes0]
        ll = lambda p: (y * np.log(p) + (n - y) * np.log(1 - p)).sum()
        lr = 2 * (ll(pm) - ll(np.full(len(y), p0)))
        resid_df = len(y) - k
        phi = (((y - n * pm) ** 2) / (n * pm * (1 - pm))).sum() / resid_df
        p_binom = stats.chi2.sf(lr, k - 1)                      # plain binomial (invalid here)
        p_qb = stats.f.sf(lr / (k - 1) / phi, k - 1, resid_df)  # quasi-binomial
        # 3) permutation test (shuffle time-of-day labels)
        def stat(c):
            a = np.bincount(c, weights=y, minlength=k); b = np.bincount(c, weights=n, minlength=k)
            return (((a - b * p0) ** 2) / (b * p0 * (1 - p0))).sum()
        obs, c, cnt = stat(codes0), codes0.copy(), 0
        for _ in range(5000):
            rng.shuffle(c); cnt += stat(c) >= obs
        rows.append({"Haul": haul, "Tier": tier, "ANOVA p": p_anova, "Binomial p (invalid)": p_binom,
                     "Dispersion": phi, "Quasi-binomial p": p_qb, "Permutation p": (cnt + 1) / 5001})

tod = pd.DataFrame(rows)
tod["Quasi-binomial Holm p"] = holm(tod["Quasi-binomial p"].values)
tod["Permutation Holm p"] = holm(tod["Permutation p"].values)
display(tod.round(4))

### Statistical Tests of Time-of-Day Differences in Eligibility Rates

ANOVA, quasi-binomial and permutation tests were used to assess whether eligibility rates differed across time-of-day groups within each haul type and tier. The quasi-binomial p-values are quoted below because that test allows for the extra flight-to-flight variation in the data.

**Key findings:**

- **Long-haul flights:** No statistically significant time-of-day differences for Tier 1 (quasi-binomial p = 0.331), Tier 2 (p = 0.475) or Tier 3 (p = 0.461).
- **Short-haul flights:** Tier 2 (p = 0.222) and Tier 3 (p = 0.250) showed no significant differences. Tier 1 was borderline before multiple-testing correction (p = 0.064).
- **Overdispersion:** Dispersion values ranged from 1.66 to 11.47 (1 would be expected under a plain binomial model), so the unadjusted binomial GLM p-values were not considered reliable.
- **Multiple-testing correction:** After Holm adjustment across the six tests, the smallest p-value was 0.387 (quasi-binomial) and 0.332 (permutation). No difference remained significant.
- **Consistency across methods:** ANOVA, quasi-binomial and permutation tests all led to the same conclusion: the data did not provide sufficient evidence of time-of-day differences in eligibility rates.

**Conclusion:** The analysis supports using haul type as the primary grouping variable in the eligibility lookup table, rather than adding time-of-day categories without demonstrated benefit. This does not prove that time of day has no effect; it means any effect is too small to detect reliably in this data. The head-to-head comparison on held-out flights (later in this notebook) checks whether adding time of day improves predictions.

# **How big are the time-of-day differences? (effect sizes)**

In [ ]:
# Time-specific Tier 3 rates with 95% CIs from a bootstrap that resamples whole dates
ORDER = ["Morning", "Lunchtime", "Afternoon", "Evening"]
B = 2000
rows = []
for haul, g in df.groupby("HAUL"):
    dates = np.sort(g["FLIGHT_DATE"].unique())
    di = g["FLIGHT_DATE"].map({d: i for i, d in enumerate(dates)}).values
    ti = pd.Categorical(g["TIME_OF_DAY"], categories=ORDER).codes
    n_mat = np.zeros((len(dates), 4)); y_mat = np.zeros((len(dates), 4))
    np.add.at(n_mat, (di, ti), g["TOTAL_SEATS"].values)
    np.add.at(y_mat, (di, ti), g["TIER3_ELIGIBLE_PAX"].values)
    boots = rng.integers(0, len(dates), size=(B, len(dates)))
    yb, nb = y_mat[boots].sum(1), n_mat[boots].sum(1)
    rate_b, all_b = yb / nb, yb.sum(1) / nb.sum(1)
    point, point_all = y_mat.sum(0) / n_mat.sum(0), y_mat.sum() / n_mat.sum()
    for j, tod_name in enumerate(ORDER):
        lo, hi = np.percentile(rate_b[:, j], [2.5, 97.5])
        dlo, dhi = np.percentile(rate_b[:, j] - all_b, [2.5, 97.5])
        rows.append({"Haul": haul, "Time of day": tod_name, "Flights": int((ti == j).sum()),
                     "Tier 3 rate %": 100 * point[j], "CI low %": 100 * lo, "CI high %": 100 * hi,
                     "Diff vs haul rate (pp)": 100 * (point[j] - point_all),
                     "Diff CI low": 100 * dlo, "Diff CI high": 100 * dhi,
                     "Pax per average flight": (point[j] - point_all) * g["TOTAL_SEATS"].mean()})
display(pd.DataFrame(rows).round(2))

### Tier 3 Eligibility Rates by Time of Day

Tier 3 eligibility rates were compared across four time-of-day periods within long-haul and short-haul flights. The analysis examined the observed rates, 95% confidence intervals, differences from each haul group's overall rate, and the estimated passenger impact per average flight.

**Key findings:**

- **Long-haul flights:** Tier 3 eligibility rates ranged from 10.26% in the evening to 10.63% in the morning. The largest positive difference from the overall haul rate was 0.16 percentage points in the morning.
- **Short-haul flights:** Rates ranged from 16.59% in the afternoon to 17.28% at lunchtime. Lunchtime had the largest positive difference, at approximately 0.42 percentage points.
- **Practical impact:** The largest estimated difference was approximately 0.76 eligible passengers per average short-haul flight at lunchtime. The remaining time periods showed smaller differences.
- **Confidence intervals:** All eight Tier 3 intervals for the difference from the haul rate included zero, so none of the Tier 3 differences was distinguishable from zero at the 5% level.
- **Tier 1 and Tier 2:** These were checked in a separate audit. Only one of the 24 tier, haul and time intervals excluded zero (short-haul Tier 1 at lunchtime, about +0.05 percentage points or 0.09 passengers per flight). With 24 intervals, about one is expected to exclude zero by chance, so treat it as a lead to watch, not a finding.

**Conclusion:** Although some time periods showed slightly higher or lower Tier 3 eligibility rates, the differences were small relative to the overall gap between haul types. The results support using haul type as the primary grouping variable in the eligibility lookup table, subject to confirmation that this simpler approach performs adequately on held-out data. Time of day may still be useful for understanding when demand occurs, even if it does not materially improve eligibility-rate estimates.

# **Final lookup table (all flights, April to October 2025)**

In [ ]:
TIERS = ["TIER1_ELIGIBLE_PAX", "TIER2_ELIGIBLE_PAX", "TIER3_ELIGIBLE_PAX"]

def build_lookup(data):
    totals = data.groupby("HAUL")[TIERS + ["TOTAL_SEATS"]].sum()
    rates = totals[TIERS].div(totals["TOTAL_SEATS"], axis=0) * 100
    rates.columns = ["Tier 1 %", "Tier 2 %", "Tier 3 %"]
    rates["Flights"] = data.groupby("HAUL").size()
    return rates

lookup = build_lookup(df)
display(lookup.round(2))

### **Key findings:**

Short-haul flights have higher eligibility rates across all three tiers. Tier 1 eligibility is 0.34% for short-haul flights compared with 0.20% for long-haul flights. Tier 2 rates are 4.40% and 2.74%, respectively, while Tier 3 rates are 16.85% and 10.47%.

The largest absolute difference occurs in Tier 3. Short-haul eligibility is 6.38 percentage points higher than long-haul eligibility.

The dataset contains 10,000 flights. Short-haul flights account for 5,975 flights (59.75%), while long-haul flights account for 4,025 (40.25%).

**Conclusion**: Haul type appears to be a useful primary variable for estimating tier eligibility. Short-haul and long-haul flights have consistently different eligibility rates, supporting separate lookup rates for each haul type. These are eligibility estimates, not measurements of actual lounge attendance.

# **Holdout test (build on April to June, test on July)**

In [ ]:
train = df[df["MONTH"].isin(["2025-04", "2025-05", "2025-06"])]
test = df[df["MONTH"] == "2025-07"]
train_lookup = build_lookup(train)

# Estimated eligible passengers = seats x rate for the flight's haul group
est = test.copy()
for i, tier_col in enumerate(TIERS, start=1):
    est[f"EST_T{i}"] = est["TOTAL_SEATS"] * est["HAUL"].map(train_lookup[f"Tier {i} %"]) / 100

rows = []
for i, tier_col in enumerate(TIERS, start=1):
    for haul, g in list(est.groupby("HAUL")) + [("ALL", est)]:
        rows.append({"Group": haul, "Tier": i, "Estimated": g[f"EST_T{i}"].sum(), "Actual": g[tier_col].sum()})
holdout = pd.DataFrame(rows)
holdout["Error_%"] = (holdout["Estimated"] - holdout["Actual"]) / holdout["Actual"] * 100
print(f"Train flights: {len(train):,}   Test flights: {len(test):,}")
display(holdout.round(1))

# Per-flight and per-day error
est["ABS_ERR_T3"] = (est["EST_T3"] - est["TIER3_ELIGIBLE_PAX"]).abs()
print("Mean absolute error per flight (Tier 3, passengers):", round(est["ABS_ERR_T3"].mean(), 1))
daily = est.groupby("FLIGHT_DATE")[["EST_T3", "TIER3_ELIGIBLE_PAX"]].sum()
daily_err = (daily["EST_T3"] - daily["TIER3_ELIGIBLE_PAX"]).abs() / daily["TIER3_ELIGIBLE_PAX"] * 100
print(f"Daily Tier 3 error: mean {daily_err.mean():.1f}%, worst {daily_err.max():.1f}%")

### **Key Findings**

* Tier 3 performed well at the aggregate level. The overall estimate was 42,294.6 compared with an actual count of 41,721, an overestimate of approximately 1.4%.

* Long-haul Tier 3 estimates were particularly close. The estimate was 17,471.2 against an actual count of 17,553, an underestimate of approximately 0.5%.

* Short-haul Tier 3 estimates were slightly high. The estimate exceeded the actual count by approximately 2.7%.

* Tier 1 showed the largest relative error. Long-haul Tier 1 was overestimated by 17.9%. However, the actual count was comparatively small, so a modest absolute difference can produce a large percentage error.

* Daily estimates were less consistent than aggregate estimates. Tier 3 had a mean daily percentage error of 4.4%, with a worst daily error of 14.3%. This suggests that good overall calibration does not guarantee equally accurate estimates on every day.


The results suggest that a haul-based lookup table can provide useful estimates of eligible passenger volumes, particularly for Tier 3 at the aggregate level. However, the variation in daily errors and the larger relative error for long-haul Tier 1 indicate that performance should be assessed separately by tier and haul type.

Flight-level errors were large relative to the size of each flight's count. The mean absolute error for Tier 3 was 12.6 passengers on flights that averaged about 30 eligible passengers, so roughly 42% of a typical flight's count. This is the main reason the lookup should be used for totals across many flights, not for predicting one flight.

Conclusion: The haul-based lookup is a promising baseline for estimating eligibility volumes in aggregate. The next section compares it with a haul-plus-time-of-day lookup on the same held-out flights.

# **Split check and head-to-head comparison: haul only vs haul plus time of day**

Both lookups are built on April to June only and applied to the same July flights. The first block also verifies the split and recalculates the holdout metrics independently of the cell above.

In [ ]:
# ---- split verification ----
MONTHS_TRAIN = ["2025-04", "2025-05", "2025-06"]
train = df[df["MONTH"].isin(MONTHS_TRAIN)].copy()
test = df[df["MONTH"] == "2025-07"].copy()

split_checks = pd.DataFrame({
    "Flights": [len(train), len(test)],
    "First date": [train["FLIGHT_DATE"].min().date(), test["FLIGHT_DATE"].min().date()],
    "Last date": [train["FLIGHT_DATE"].max().date(), test["FLIGHT_DATE"].max().date()],
}, index=["Train (Apr-Jun)", "Test (Jul)"])
display(split_checks)
print("Train and test share no dates:", set(train["FLIGHT_DATE"]).isdisjoint(set(test["FLIGHT_DATE"])))
print("Train + test flights:", len(train) + len(test), "| July-only test, no flight appears in both:",
      train.index.intersection(test.index).empty)

# ---- build both lookups on TRAIN only, apply to the same TEST flights ----
def build_rates(data, keys):
    totals = data.groupby(keys)[TIERS + ["TOTAL_SEATS"]].sum()
    return totals[TIERS].div(totals["TOTAL_SEATS"], axis=0)       # rates as fractions

def predict(test_df, rates, keys):
    r = test_df[keys].merge(rates, left_on=keys, right_index=True, how="left")
    assert r[TIERS].notna().all().all(), "a test group is missing from the training lookup"
    return r[TIERS].mul(test_df["TOTAL_SEATS"].values, axis=0).values

haul_only = predict(test, build_rates(train, ["HAUL"]), ["HAUL"])
haul_time = predict(test, build_rates(train, ["HAUL", "TIME_OF_DAY"]), ["HAUL", "TIME_OF_DAY"])
actual = test[TIERS].values

# ---- metrics: total bias, flight-level MAE/RMSE, daily error ----
def metrics(pred, name):
    rows = []
    for j, tier in enumerate(["Tier 1", "Tier 2", "Tier 3"]):
        err = pred[:, j] - actual[:, j]
        daily = pd.DataFrame({"p": pred[:, j], "a": actual[:, j], "d": test["FLIGHT_DATE"].values}).groupby("d").sum()
        daily_pct = ((daily["p"] - daily["a"]).abs() / daily["a"].replace(0, np.nan) * 100)
        rows.append({"Model": name, "Tier": tier,
                     "Total error %": (pred[:, j].sum() - actual[:, j].sum()) / actual[:, j].sum() * 100,
                     "Flight MAE": np.abs(err).mean(), "Flight RMSE": np.sqrt((err ** 2).mean()),
                     "Daily MAPE %": daily_pct.mean(), "Worst day %": daily_pct.max()})
    return pd.DataFrame(rows)

comparison = pd.concat([metrics(haul_only, "Haul only"), metrics(haul_time, "Haul + time of day")])
display(comparison.sort_values(["Tier", "Model"]).round(2))

# ---- is the difference real? paired date-cluster bootstrap of the flight-level MAE difference ----
rng2 = np.random.default_rng(7)
dates_u = np.sort(test["FLIGHT_DATE"].unique())
di = test["FLIGHT_DATE"].map({d: i for i, d in enumerate(dates_u)}).values
rows = []
for j, tier in enumerate(["Tier 1", "Tier 2", "Tier 3"]):
    diff = np.abs(haul_only[:, j] - actual[:, j]) - np.abs(haul_time[:, j] - actual[:, j])   # positive = time-of-day model better
    s = np.bincount(di, weights=diff, minlength=len(dates_u)); c = np.bincount(di, minlength=len(dates_u))
    boot = rng2.integers(0, len(dates_u), size=(2000, len(dates_u)))
    m = s[boot].sum(1) / c[boot].sum(1)
    rows.append({"Tier": tier, "MAE gain from adding time of day": diff.mean(),
                 "95% CI low": np.percentile(m, 2.5), "95% CI high": np.percentile(m, 97.5)})
display(pd.DataFrame(rows).round(3))
print("A positive gain means the haul + time-of-day lookup had smaller errors. If the interval includes 0, the extra rows did not clearly help.")

### **Results: does adding time of day help?**

**Split check:** 4,407 training flights (1 April to 30 June) and 1,389 test flights (1 to 30 July). The two sets share no dates and no flights, so the test is genuinely out of sample. The recalculated totals match the holdout above (total error 3.4% Tier 1, 1.8% Tier 2, 1.4% Tier 3 for the haul-only lookup).

**Head-to-head:** The haul-plus-time-of-day lookup performed essentially the same as the haul-only lookup on every measure.

- **Flight-level error:** Tier 3 mean absolute error was 12.57 passengers (haul plus time) against 12.58 (haul only). Tier 2 was 4.14 against 4.15 and Tier 1 was 0.77 against 0.76.
- **Total error:** Differences were 0.1 percentage points or less in Tiers 2 and 3 (the time-of-day lookup was marginally closer: 1.7% against 1.8% and 1.3% against 1.4%), and slightly worse for Tier 1 with the extra rows (3.7% against 3.4%).
- **Daily error:** The time-of-day lookup was marginally worse for Tier 3 (4.48% against 4.42% mean daily error).
- **Is the gain real?** The average improvement in Tier 3 flight-level error from adding time of day was 0.010 passengers, with a 95% interval of -0.026 to 0.047 that comfortably includes zero. Tier 1 and Tier 2 intervals also include zero.

**Conclusion:** Adding time-of-day rows does not improve predictions on unseen flights, in any tier, at flight or daily level. This supports the haul-only table: it is simpler and no less accurate. Time of day remains useful for planning when demand arrives, but not for the eligibility rates.

# **Worked example (morning departures on 15 July 2025)**

In [ ]:
example = df[(df["FLIGHT_DATE"] == "2025-07-15")].copy()
example["HOUR"] = pd.to_datetime(example["FLIGHT_TIME"].astype(str), format="%H:%M:%S").dt.hour
example = example[example["HOUR"].between(6, 11)].sort_values("FLIGHT_TIME")

for i, tier_col in enumerate(TIERS, start=1):
    example[f"Est Tier {i}"] = example["TOTAL_SEATS"] * example["HAUL"].map(lookup[f"Tier {i} %"]) / 100

cols = ["FLIGHT_NO", "FLIGHT_TIME", "ARRIVAL_STATION_CD", "HAUL", "TOTAL_SEATS",
        "Est Tier 1", "Est Tier 2", "Est Tier 3"] + TIERS
display(example[cols].round(1))
print(example[["Est Tier 1", "Est Tier 2", "Est Tier 3"] + TIERS].sum().round(1))

### **Worked Example Review: Morning Departures on 15 July 2025 (20 flights)**

This example applies the final haul-based lookup (built on all flights from April to October) to the 20 departures between 06:00 and 11:59 on 15 July 2025. Because July is part of the data that built these rates, it is an **illustration, not an out-of-sample test**. The out-of-sample evidence is the April-June to July holdout above.

#### **Key Findings**

- **Tier 2 estimates were highly accurate in aggregate.** The estimated total of 155.6 was very close to the observed total of 156.
- **Tier 3 estimates were also close to the observed total.** The lookup underestimated the actual count by 11.8 passengers, about 1.9%.
- **Tier 1 showed the largest relative error.** The estimate was 11.9 against 17 actual, an underestimate of about 30%. With so few Tier 1 passengers, a few passengers move the percentage a lot, and this 20-flight window should not be read as a Tier 1 performance measure (the full July holdout overestimated Tier 1 by 3.4%).
- **Individual flights varied considerably.** Flight BA4465 had 41 observed Tier 3 eligible passengers against an estimate of 30.3, while BA4918 recorded only 2 against an estimate of 22.4.

#### **Interpretation**

The haul-based lookup gives reasonable totals for Tiers 2 and 3 across these 20 flights, but it does not capture each flight's eligibility count. Close totals can hide large errors on individual flights, which is why the table should be used for aggregates.

# **Recommended Lounge Eligibility Lookup: Findings and Business Recommendations**

## 1. **Recommended Table: Group by Haul Type**

Use **haul type as the only grouping variable**, with separate rates for each tier. Rates are eligible passengers as a share of seats.

| Haul type | Tier 1 | Tier 2 | Tier 3 |
|---|---|---|---|
| Short-haul (Europe) | 0.34% | 4.40% | 16.85% |
| Long-haul (North America, Middle East, Asia) | 0.20% | 2.74% | 10.47% |

**Why haul type only?**

- Short-haul flights have higher eligibility rates in all three tiers (Tier 3 is 6.38 percentage points higher).
- Region adds nothing beyond haul: every short-haul flight goes to Europe, and the three long-haul regions are within about 0.3 percentage points of each other.
- Time-of-day differences were not statistically significant after accounting for overdispersion and multiple comparisons, and the largest was about 0.4 percentage points (under 1 passenger per average flight).
- On the same held-out July flights, a haul-plus-time-of-day lookup was no more accurate than the haul-only lookup in any tier.

## 2. **What the Holdout Test Showed**

Rates were built on April to June (4,407 flights) and tested on July (1,389 flights). Positive errors indicate overestimation.

- **Tier 3:** The overall estimate was 1.4% above the observed total (0.5% under for long-haul, 2.7% over for short-haul).
- **Tier 2:** The overall estimate was 1.8% above the observed total.
- **Tier 1:** The overall estimate was 3.4% above, but long-haul Tier 1 was 17.9% too high. Tier 1 counts are small, so a modest absolute difference produces a large percentage error.
- **Daily and flight-level variation:** The mean daily Tier 3 error was 4.4% (worst day 14.3%). The mean flight-level Tier 3 error was 12.6 passengers, about 42% of a typical flight's count.

**Interpretation:** The lookup is dependable for totals across groups, days or time windows, and not for individual flights.

## 3. **Assumptions and Limitations**

1. **Seats are not actual passengers.** The method uses total seats, so it does not account for load factors and is not a direct estimate of eligible passengers who will travel or enter a lounge.
2. **Tiers are reported separately.** The data shows the tiers are not nested (Tier 1 exceeds Tier 2 on 459 flights and Tier 3 on 126) but does not establish whether passengers overlap between tiers, so the tiers should not be added together to get a unique passenger count.
3. **Future flight mix may change.** Applying historical rates (April to October 2025) to a future period assumes that the mix of haul types, aircraft and seat capacities, and passenger eligibility patterns stays similar.
4. **Connecting passengers are not modelled.** The table covers eligibility on the flights in the dataset only.
5. **Lounge dwell time is not modelled.** Eligible passengers per flight do not determine how many people are in a lounge at once; arrival timing, time spent in the lounge and overlapping flows would be needed for occupancy planning.
6. **Accuracy varies by level.** Aggregate estimates can be close while flight-level or daily errors are large, so evaluate at the level at which BA will use the estimates.

## 4. **How BA Should Use the Lookup**

- **Aggregate planning:** Estimate eligible passenger volumes across multiple flights, days or time windows.
- **Initial planning input only:** The results are not a measure of lounge occupancy or a guarantee of demand.
- **Avoid single-flight estimates:** Flight-level errors are large.
- **Monitor errors by tier and haul type:** Track estimated against observed counts, especially long-haul Tier 1 and days with large Tier 3 errors.
- **Refresh rates when the mix changes:** Recalculate when aircraft and seat capacity, loyalty composition or route mix change.
- **Use time of day for timing, not rates:** Time of day can help forecast when demand arrives, not how many passengers are eligible.

### **Final recommendation**

Adopt the haul-only lookup as a simple baseline for aggregate eligibility estimation. The split was verified (April to June training, July test, no overlap), and the haul-plus-time-of-day lookup showed no improvement on the same test flights. Retain separate rates for all three tiers, state the limitations clearly, and add other groupings only if they improve out-of-sample accuracy.